In [0]:
silver_path = "/Volumes/workspace/default/retail_volume/silver"

silver_df = spark.read.format("delta").load(silver_path)

print("Rows:", silver_df.count())

Rows: 26157


In [0]:
from pyspark.sql.functions import to_timestamp

gold_df = silver_df.withColumn(
    "InvoiceDate",
    to_timestamp("InvoiceDate")
)

display(gold_df.limit(5))

InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue
536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6.0,2010-12-01T08:26:00.000Z,2.55,17850.0,United Kingdom,15.299999999999999
536365,71053,WHITE METAL LANTERN,6.0,2010-12-01T08:26:00.000Z,3.39,17850.0,United Kingdom,20.34
536365,84406B,CREAM CUPID HEARTS COAT HANGER,8.0,2010-12-01T08:26:00.000Z,2.75,17850.0,United Kingdom,22.0
536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6.0,2010-12-01T08:26:00.000Z,3.39,17850.0,United Kingdom,20.34
536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6.0,2010-12-01T08:26:00.000Z,3.39,17850.0,United Kingdom,20.34


In [0]:
from pyspark.sql.functions import weekofyear, year, sum

weekly_revenue = (
    gold_df
    .groupBy(
        year("InvoiceDate").alias("Year"),
        weekofyear("InvoiceDate").alias("Week")
    )
    .agg(
        sum("Revenue").alias("TotalRevenue")
    )
    .orderBy("Year", "Week")
)

display(weekly_revenue)

Year,Week,TotalRevenue
2010,48,149386.33000000095
2010,49,213447.7200000031
2010,50,163770.7300000008
2010,51,46109.11000000002


In [0]:
from pyspark.sql.functions import desc

top_products = (
    gold_df
    .groupBy("Description")
    .agg(
        sum("Revenue").alias("TotalRevenue")
    )
    .orderBy(desc("TotalRevenue"))
    .limit(5)
)

display(top_products)

Description,TotalRevenue
REGENCY CAKESTAND 3 TIER,17756.700000000008
WHITE HANGING HEART T-LIGHT HOLDER,9602.050000000005
VINTAGE UNION JACK MEMOBOARD,6938.489999999996
WOOD BLACK BOARD ANT WHITE FINISH,6685.2300000000005
BLACK RECORD COVER FRAME,6248.820000000001


In [0]:
top_customers = (
    gold_df
    .groupBy("CustomerID")
    .agg(
        sum("Revenue").alias("TotalRevenue")
    )
    .orderBy(desc("TotalRevenue"))
    .limit(5)
)

display(top_customers)

CustomerID,TotalRevenue
18102.0,27834.61
15061.0,19950.660000000007
16029.0,13112.52
14646.0,8591.879999999997
14911.0,7737.939999999999


In [0]:
country_revenue = (
    gold_df
    .groupBy("Country")
    .agg(
        sum("Revenue").alias("TotalRevenue")
    )
    .orderBy(desc("TotalRevenue"))
)

display(country_revenue)

Country,TotalRevenue
United Kingdom,498661.8500000227
Germany,15241.139999999996
France,9616.30999999999
EIRE,8813.880000000001
Netherlands,8784.48
Japan,7705.0700000000015
Sweden,3834.3
Norway,3787.1199999999994
Portugal,2439.9699999999984
Spain,1843.7299999999998


In [0]:
weekly_path = "/Volumes/workspace/default/retail_volume/gold/weekly_revenue"

weekly_revenue.write \
    .format("delta") \
    .mode("overwrite") \
    .save(weekly_path)

print("Weekly revenue saved")

Weekly revenue saved


In [0]:
products_path = "/Volumes/workspace/default/retail_volume/gold/top_products"

top_products.write \
    .format("delta") \
    .mode("overwrite") \
    .save(products_path)

print("Top products saved")

Top products saved


In [0]:
customers_path = "/Volumes/workspace/default/retail_volume/gold/top_customers"

top_customers.write \
    .format("delta") \
    .mode("overwrite") \
    .save(customers_path)

print("Top customers saved")

Top customers saved


In [0]:
country_path = "/Volumes/workspace/default/retail_volume/gold/country_revenue"

country_revenue.write \
    .format("delta") \
    .mode("overwrite") \
    .save(country_path)

print("Country revenue saved")

Country revenue saved


In [0]:
display(
    dbutils.fs.ls(
        "/Volumes/workspace/default/retail_volume/gold"
    )
)

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/retail_volume/gold/country_revenue/,country_revenue/,0,1782285376253
dbfs:/Volumes/workspace/default/retail_volume/gold/top_customers/,top_customers/,0,1782285376253
dbfs:/Volumes/workspace/default/retail_volume/gold/top_products/,top_products/,0,1782285376253
dbfs:/Volumes/workspace/default/retail_volume/gold/weekly_revenue/,weekly_revenue/,0,1782285376253
